# Optimization Foundations — Hands-on

## 1. Gradient descent on a toy loss function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(w):
    return (w - 3)**2 + 2

def grad(w):
    return 2 * (w - 3)

w = -5.0
lr = 0.2
path = [w]

for _ in range(25):
    w = w - lr * grad(w)
    path.append(w)

ws = np.linspace(-6, 10, 200)
plt.plot(ws, loss(ws), label="loss surface")
plt.plot(path, [loss(p) for p in path], 'ro-', label="descent path")
plt.legend()
plt.title("Gradient descent converging on a toy loss")
plt.show()

## 2. Batch vs Stochastic vs Mini-batch gradient descent

In [ ]:
import numpy as np

np.random.seed(1)
X = np.random.randn(100, 1)
true_w = 4.0
y = true_w * X.flatten() + np.random.randn(100) * 0.5

def compute_grad(X_batch, y_batch, w):
    preds = w * X_batch.flatten()
    error = preds - y_batch
    return np.mean(2 * error * X_batch.flatten())

def run_gd(batch_size, steps=50, lr=0.1):
    w = 0.0
    n = len(X)
    for step in range(steps):
        idx = np.random.choice(n, batch_size, replace=False)
        g = compute_grad(X[idx], y[idx], w)
        w -= lr * g
    return w

for bs, label in [(100, "batch"), (1, "stochastic"), (16, "mini-batch")]:
    w_final = run_gd(bs)
    print(f"{label} (batch_size={bs}): learned w = {w_final:.3f} (true w = {true_w})")

## 3. Learning rate sensitivity

In [ ]:
def gd_run(lr, steps=15, start=-5.0):
    w = start
    path = [w]
    for _ in range(steps):
        w = w - lr * grad(w)
        path.append(w)
    return path

for lr, label in [(0.01, "too low"), (0.2, "good"), (1.1, "too high (diverges)")]:
    path = gd_run(lr)
    print(f"lr={lr} ({label}): final w = {path[-1]:.3f}")

## 4. Convex vs non-convex loss surfaces

In [ ]:
def convex_loss(w):
    return (w - 3)**2 + 2

def nonconvex_loss(w):
    # Multiple local minima
    return 0.3*(w**4) - 2*(w**2) + 0.5*w + 5

ws = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ws, convex_loss(ws))
axes[0].set_title("Convex: one global minimum")

axes[1].plot(ws, nonconvex_loss(ws))
axes[1].set_title("Non-convex: multiple local minima")
plt.show()

## 5. Getting stuck in a local minimum (non-convex case)

In [ ]:
def grad_nonconvex(w):
    return 1.2*(w**3) - 4*w + 0.5

# Two different starting points can converge to different minima
for start, label in [(-3.0, "start far left"), (0.5, "start near center")]:
    w = start
    for _ in range(50):
        w = w - 0.05 * grad_nonconvex(w)
    print(f"{label}: converged to w = {w:.3f}, loss = {nonconvex_loss(w):.3f}")

## 6. Plain gradient descent vs momentum

In [ ]:
def gd_plain(lr=0.1, steps=30, start=-5.0):
    w = start
    path = [w]
    for _ in range(steps):
        w = w - lr * grad(w)
        path.append(w)
    return path

def gd_momentum(lr=0.1, beta=0.9, steps=30, start=-5.0):
    w = start
    v = 0.0
    path = [w]
    for _ in range(steps):
        v = beta * v + (1 - beta) * grad(w)
        w = w - lr * v
        path.append(w)
    return path

path_plain = gd_plain()
path_momentum = gd_momentum()

plt.plot([loss(p) for p in path_plain], 'o-', label='plain GD')
plt.plot([loss(p) for p in path_momentum], 's-', label='GD with momentum')
plt.xlabel('step')
plt.ylabel('loss')
plt.legend()
plt.title('Momentum converges faster by smoothing the update direction')
plt.show()